In [1]:
!pip install torch_geometric
!pip install torch_scatter torch_sparse -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__.split('+')[0])")+cu$(python -c "import torch; print(torch.version.cuda.replace('.', ''))").html
!pip install rdkit nltk six

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.8 MB/s eta 0:00:0000:01
Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 88.5 MB/s eta 0:00:0000:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 104.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 54.5 MB/s eta 0:00:00:00:0100:01


In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
experiment_no_bias_arch_tuned.py
=================================

Critical missing experiment: No-Bias + Architecture Tuning

This script tests whether the property-aware electron_bias provides any benefit
when the architecture is properly tuned (3 GCN layers, 0.2 dropout).

Comparison:
  Variant 4 (Arch Tuned):     electron_bias=0.1, gcn_layers=3, dropout=0.2 → RMSE=0.46546
  Variant 5 (This script):    electron_bias=0.0, gcn_layers=3, dropout=0.2 → RMSE=???

If Variant 5 performs BETTER than Variant 4, it proves electron_bias hurts performance
even with optimal hyperparameters.

If Variant 5 performs WORSE than Variant 4, it proves electron_bias helps when
architecture is properly configured.

Dataset: ESOL (Delaney Aqueous Solubility)
Hardware: GPU recommended (CUDA if available)
Runtime: ~65 minutes
"""

import os
import sys
import time
import math
import subprocess

# ============================================================
# DEPENDENCY INSTALLATION
# ============================================================

# ============================================================
# IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION
# ============================================================

# Path to ESOL CSV file
# Must have columns: 'smiles' and 'measured log solubility in mols per litre'
ESOL_CSV_PATH = '/kaggle/input/datasets/kaalmurlidhar/esolandfreesol/delaney-processed.csv'  # Update this path

# Experiment configuration
EXPERIMENT_NAME = "Variant 5: No-Bias + Arch Tuned"
ELECTRON_BIAS_INIT = 0.0      # No bias
ELECTRON_BIAS_LEARNABLE = False  # Frozen at 0.0
GCN_LAYERS = 3                # Tuned (reduced from 5)
PREDICTOR_DROPOUT = 0.2       # Tuned (increased from 0.1)

# Training hyperparameters (match original ESOL settings)
EPOCHS = 100
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.0
SEED = 2024
EARLY_STOP_PATIENCE = 50
LR_FACTOR = 0.6
LR_PATIENCE = 10
LR_MIN = 5e-6

# Model architecture (match original)
D_MODEL = 768
N_LAYERS = 6
N_HEADS = 12
D_K = 64
D_V = 64
D_FF = 768 * 4
VOCAB_SIZE = 83
MAXLEN = 501
GCN_EMB_DIM = 300
GCN_FEAT_DIM = 300
COMBINED_DIM = D_MODEL + GCN_FEAT_DIM

# Set random seeds
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"\n{'='*64}")
print(f"{EXPERIMENT_NAME}")
print(f"{'='*64}")
print(f"Configuration:")
print(f"  electron_bias_init : {ELECTRON_BIAS_INIT}")
print(f"  electron_bias_learnable: {ELECTRON_BIAS_LEARNABLE}")
print(f"  gcn_layers         : {GCN_LAYERS}")
print(f"  predictor_dropout  : {PREDICTOR_DROPOUT}")
print(f"{'='*64}\n")

# ============================================================
# GRAMMAR AND ENCODING
# ============================================================

import nltk
import re as _re

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

# ZINC CFG Grammar for SMILES parsing
GRAM_STRING = """smiles -> chain
atom -> bracket_atom
atom -> aliphatic_organic
atom -> aromatic_organic
aliphatic_organic -> 'B'
aliphatic_organic -> 'C'
aliphatic_organic -> 'N'
aliphatic_organic -> 'O'
aliphatic_organic -> 'S'
aliphatic_organic -> 'P'
aliphatic_organic -> 'F'
aliphatic_organic -> 'I'
aliphatic_organic -> 'Cl'
aliphatic_organic -> 'Br'
aromatic_organic -> 'c'
aromatic_organic -> 'n'
aromatic_organic -> 'o'
aromatic_organic -> 's'
aromatic_organic -> 'p'
bracket_atom -> '[' BAI ']'
BAI -> isotope symbol BAC
BAI -> symbol BAC
BAI -> isotope symbol
BAI -> symbol
BAC -> chiral BAH
BAC -> BAH
BAC -> chiral
BAH -> hcount BACH
BAH -> BACH
BAH -> hcount
BACH -> charge class
BACH -> charge
BACH -> class
symbol -> aliphatic_organic
symbol -> aromatic_organic
symbol -> 'H'
isotope -> DIGIT
isotope -> DIGIT DIGIT
isotope -> DIGIT DIGIT DIGIT
DIGIT -> '0'
DIGIT -> '1'
DIGIT -> '2'
DIGIT -> '3'
DIGIT -> '4'
DIGIT -> '5'
DIGIT -> '6'
DIGIT -> '7'
DIGIT -> '8'
DIGIT -> '9'
charge -> '-'
charge -> '-' DIGIT
charge -> '-' DIGIT DIGIT
charge -> '+'
charge -> '+' DIGIT
charge -> '+' DIGIT DIGIT
class -> DIGIT
bond -> '-'
bond -> '='
bond -> '#'
bond -> '/'
bond -> '\\\\'
bond -> '.'
ringbond -> DIGIT
ringbond -> bond DIGIT
branched_atom -> atom
branched_atom -> atom RB
branched_atom -> atom BB
branched_atom -> atom RB BB
RB -> RB ringbond
RB -> ringbond
BB -> BB branch
BB -> branch
branch -> '(' chain ')'
branch -> '(' bond chain ')'
chain -> branched_atom
chain -> chain branched_atom
chain -> chain bond branched_atom
chiral -> '@'
chiral -> '@@'
hcount -> 'H'
hcount -> 'H' DIGIT
"""

# Parse grammar and build lookup table
GCFG = nltk.CFG.fromstring(GRAM_STRING)
_RULE_LOOKUP = {}
for _idx, _prod in enumerate(GCFG.productions()):
    _lhs = _prod.lhs().symbol()
    _rhs = tuple(s.symbol() if not isinstance(s, str) else s for s in _prod.rhs())
    _RULE_LOOKUP[(_lhs, _rhs)] = _idx + 1

_PARSER = nltk.ChartParser(GCFG)
_SMILES_RE = _re.compile(r'(\[[^\]]+\]|Br|Cl|@@|[BCNOPSFIbcnosp=#@+\-\/\\\.1-9])')
_GRAMMAR_TOKEN_SET = set(range(1, 81))


def _tokenize(smiles):
    """Tokenize SMILES string into grammar tokens"""
    return _SMILES_RE.findall(smiles)


def encode_smiles(smiles):
    """Encode SMILES string into list of 1-based grammar rule integers"""
    tokens = _tokenize(smiles)
    if not tokens:
        raise ValueError(f'tokenizer returned empty: {smiles}')
    trees = list(_PARSER.parse(tokens))
    if not trees:
        raise ValueError(f'cfg parse failed: {smiles}')
    rule_indices = []
    for subtree in trees[0].subtrees():
        if subtree.height() > 1:
            lhs = subtree.label()
            rhs = tuple(child.label() if isinstance(child, nltk.Tree) else child for child in subtree)
            idx = _RULE_LOOKUP.get((lhs, rhs))
            if idx is not None:
                rule_indices.append(idx)
    return rule_indices


def construct_token_sequence(rule_index_list, max_len=500):
    """Construct token sequence with [GLO] token and padding"""
    if len(rule_index_list) > max_len:
        rule_index_list = rule_index_list[:max_len]
    tokens = ['[GLO]'] + rule_index_list + ['[PAD]'] * (max_len - len(rule_index_list))
    tokens_idx, atom_mask = [], []
    for token in tokens:
        if token in _GRAMMAR_TOKEN_SET:
            atom_mask.append(1)
            tokens_idx.append(token)
        elif token == '[GLO]':
            atom_mask.append(1)
            tokens_idx.append(82)
        else:
            atom_mask.append(0)
            tokens_idx.append(0)
    return np.array(tokens_idx), np.array(atom_mask)


# RDKit imports and setup
from rdkit import Chem
from rdkit.Chem.rdchem import BondType as BT
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

_ATOM_LIST = list(range(1, 119))
_CHIRALITY_LIST = [
    Chem.rdchem.ChiralType.CHI_UNSPECIFIED,
    Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CW,
    Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CCW,
    Chem.rdchem.ChiralType.CHI_OTHER,
]
_BOND_LIST = [BT.SINGLE, BT.DOUBLE, BT.TRIPLE, BT.AROMATIC]
_BONDDIR_LIST = [
    Chem.rdchem.BondDir.NONE,
    Chem.rdchem.BondDir.ENDUPRIGHT,
    Chem.rdchem.BondDir.ENDDOWNRIGHT,
]


def smiles_to_graph(smiles):
    """Convert SMILES to graph representation (x, edge_index, edge_attr)"""
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    type_idx, chirality_idx = [], []
    for atom in mol.GetAtoms():
        try:
            type_idx.append(_ATOM_LIST.index(atom.GetAtomicNum()))
        except:
            type_idx.append(0)
        try:
            chirality_idx.append(_CHIRALITY_LIST.index(atom.GetChiralTag()))
        except:
            chirality_idx.append(0)
    
    x = torch.cat([
        torch.tensor(type_idx, dtype=torch.long).view(-1, 1),
        torch.tensor(chirality_idx, dtype=torch.long).view(-1, 1),
    ], dim=-1)
    
    row, col, edge_feat = [], [], []
    for bond in mol.GetBonds():
        s, e = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        row += [s, e]
        col += [e, s]
        try:
            bt = _BOND_LIST.index(bond.GetBondType())
        except:
            bt = 0
        try:
            bd = _BONDDIR_LIST.index(bond.GetBondDir())
        except:
            bd = 0
        edge_feat += [[bt, bd], [bt, bd]]
    
    if not row:
        row, col = [0], [0]
        edge_feat = [[4, 0]]
    
    edge_index = torch.tensor([row, col], dtype=torch.long)
    edge_attr = torch.tensor(edge_feat, dtype=torch.long)
    
    return x, edge_index, edge_attr

# ============================================================
# DATASET
# ============================================================

class ESOLDataset(Dataset):
    def __init__(self, csv_path, smiles_col='smiles', 
                 target_col='measured log solubility in mols per litre'):
        df = pd.read_csv(csv_path)
        self.smiles = df[smiles_col].tolist()
        self.targets = df[target_col].tolist()
        
        # Filter valid molecules
        valid_indices = []
        for i in range(len(self.smiles)):
            try:
                # Test encoding
                rules = encode_smiles(self.smiles[i])
                construct_token_sequence(rules)
                smiles_to_graph(self.smiles[i])
                valid_indices.append(i)
            except:
                pass
        
        self.smiles = [self.smiles[i] for i in valid_indices]
        self.targets = [self.targets[i] for i in valid_indices]
        print(f"Loaded {len(self.smiles)} valid molecules")
    
    def __len__(self):
        return len(self.smiles)
    
    def __getitem__(self, idx):
        smiles = self.smiles[idx]
        target = self.targets[idx]
        
        # Grammar encoding
        rules = encode_smiles(smiles)
        tokens_idx, atom_mask = construct_token_sequence(rules)
        
        # Graph features
        x, edge_index, edge_attr = smiles_to_graph(smiles)
        
        return {
            'tokens_idx': torch.tensor(tokens_idx, dtype=torch.long),
            'atom_mask': torch.tensor(atom_mask, dtype=torch.long),
            'target': torch.tensor([target], dtype=torch.float),
            'x': x,
            'edge_index': edge_index,
            'edge_attr': edge_attr
        }

def collate_fn(batch):
    """Collate function for batching"""
    tokens_idx = torch.stack([item['tokens_idx'] for item in batch])
    atom_mask = torch.stack([item['atom_mask'] for item in batch])
    targets = torch.stack([item['target'] for item in batch])
    
    # Concatenate graphs
    x_list, edge_index_list, edge_attr_list, batch_indices = [], [], [], []
    num_nodes = 0
    for i, item in enumerate(batch):
        x_list.append(item['x'])
        edge_index_list.append(item['edge_index'] + num_nodes)
        edge_attr_list.append(item['edge_attr'])
        batch_indices.append(torch.full((item['x'].size(0),), i, dtype=torch.long))
        num_nodes += item['x'].size(0)
    
    x = torch.cat(x_list, dim=0)
    edge_index = torch.cat(edge_index_list, dim=1)
    edge_attr = torch.cat(edge_attr_list, dim=0)
    batch_idx = torch.cat(batch_indices, dim=0)
    
    return tokens_idx, atom_mask, targets, x, edge_index, edge_attr, batch_idx

# ============================================================
# MODEL ARCHITECTURE
# ============================================================

ELECTRON_RELEVANT_TOKENS = {7, 8, 9, 15, 16, 17, 18, 57, 58, 78}

def get_attn_pad_mask(seq_q):
    batch_size, seq_len = seq_q.size()
    pad_attn_mask = seq_q.data.eq(0).unsqueeze(1)
    return pad_attn_mask.expand(batch_size, seq_len, seq_len)

class Embedding(nn.Module):
    """
    Property-aware embedding with configurable electron_bias.
    
    For this experiment:
    - electron_bias initialized to 0.0
    - electron_bias frozen (not learnable)
    - This effectively disables the property-aware modification
    """
    def __init__(self, vocab_size, d_model, maxlen, bias_init=0.0, bias_learnable=False):
        super().__init__()
        self.tok_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(maxlen, d_model)
        self.norm = nn.LayerNorm(d_model)
        
        # Electron bias parameter
        if bias_learnable:
            self.electron_bias = nn.Parameter(torch.tensor(bias_init))
        else:
            self.register_buffer('electron_bias', torch.tensor(bias_init))
        
        self.register_buffer(
            'electron_token_indices',
            torch.tensor(sorted(ELECTRON_RELEVANT_TOKENS), dtype=torch.long)
        )
    
    def forward(self, x):
        seq_len = x.size(1)
        pos = torch.arange(seq_len, dtype=torch.long).unsqueeze(0).expand_as(x).to(x.device)
        
        tok_embedding = self.tok_embed(x)
        pos_embedding = self.pos_embed(pos)
        
        # Build electron mask
        electron_mask = (x.unsqueeze(-1) == self.electron_token_indices).any(dim=-1)
        electron_mask = electron_mask.unsqueeze(-1).float()
        
        # Apply bias (will be 0.0 * 0.0 = 0.0 in this experiment)
        electron_bump = electron_mask * self.electron_bias
        
        return self.norm(tok_embedding + pos_embedding + electron_bump)

class ScaledDotProductAttention(nn.Module):
    def __init__(self, d_k):
        super().__init__()
        self.d_k = d_k
    
    def forward(self, Q, K, V, attn_mask):
        scores = torch.matmul(Q, K.transpose(-1, -2)) / math.sqrt(self.d_k)
        scores.masked_fill_(attn_mask, -1e9)
        attn = F.softmax(scores, dim=-1)
        return torch.matmul(attn, V)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, d_k, d_v, n_heads):
        super().__init__()
        self.d_k = d_k
        self.d_v = d_v
        self.n_heads = n_heads
        self.W_Q = nn.Linear(d_model, d_k * n_heads)
        self.W_K = nn.Linear(d_model, d_k * n_heads)
        self.W_V = nn.Linear(d_model, d_v * n_heads)
        self.linear = nn.Linear(n_heads * d_v, d_model)
        self.layernorm = nn.LayerNorm(d_model)
    
    def forward(self, Q, K, V, attn_mask):
        residual, batch_size = Q, Q.size(0)
        q_s = self.W_Q(Q).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        k_s = self.W_K(K).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        v_s = self.W_V(V).view(batch_size, -1, self.n_heads, self.d_v).transpose(1, 2)
        attn_mask = attn_mask.unsqueeze(1).repeat(1, self.n_heads, 1, 1)
        context = ScaledDotProductAttention(self.d_k)(q_s, k_s, v_s, attn_mask)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.n_heads * self.d_v)
        return self.layernorm(self.linear(context) + residual)

class PoswiseFeedForwardNet(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(d_model, d_ff, bias=False),
            nn.ReLU(),
            nn.Linear(d_ff, d_model, bias=False),
        )
        self.layernorm = nn.LayerNorm(d_model)
    
    def forward(self, inputs):
        return self.layernorm(self.fc(inputs) + inputs)

class EncoderLayer(nn.Module):
    def __init__(self, d_model, d_k, d_v, n_heads, d_ff):
        super().__init__()
        self.enc_self_attn = MultiHeadAttention(d_model, d_k, d_v, n_heads)
        self.pos_ffn = PoswiseFeedForwardNet(d_model, d_ff)
    
    def forward(self, enc_inputs, enc_self_attn_mask):
        enc_outputs = self.enc_self_attn(enc_inputs, enc_inputs, enc_inputs, enc_self_attn_mask)
        return self.pos_ffn(enc_outputs)

class TransformerEncoder(nn.Module):
    def __init__(self, d_model, n_layers, vocab_size, maxlen, d_k, d_v, n_heads, d_ff,
                 bias_init=0.0, bias_learnable=False):
        super().__init__()
        self.embedding = Embedding(vocab_size, d_model, maxlen, bias_init, bias_learnable)
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, d_k, d_v, n_heads, d_ff) for _ in range(n_layers)
        ])
    
    def forward(self, input_ids):
        output = self.embedding(input_ids)
        attn_mask = get_attn_pad_mask(input_ids)
        for layer in self.layers:
            output = layer(output, attn_mask)
        return output[:, 0, :]  # [GLO] token

# ============================================================
# GCN IMPLEMENTATION
# ============================================================

from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops
from torch_scatter import scatter_add
import torch_sparse

_NUM_ATOM_TYPE = 119
_NUM_CHIRALITY_TAG = 3
_NUM_BOND_TYPE = 5
_NUM_BOND_DIRECTION = 3


def _gcn_norm(edge_index, num_nodes=None):
    """GCN normalization"""
    from torch_geometric.utils.num_nodes import maybe_num_nodes
    num_nodes = maybe_num_nodes(edge_index, num_nodes)
    edge_weight = torch.ones(edge_index.size(1), device=edge_index.device)
    row, col = edge_index[0], edge_index[1]
    deg = scatter_add(edge_weight, col, dim=0, dim_size=num_nodes)
    deg_inv_sqrt = deg.pow_(-0.5)
    deg_inv_sqrt.masked_fill_(deg_inv_sqrt == float('inf'), 0)
    return edge_index, deg_inv_sqrt[row] * edge_weight * deg_inv_sqrt[col]


class GCNConv(MessagePassing):
    def __init__(self, emb_dim, aggr='add'):
        super().__init__()
        self.emb_dim = emb_dim
        self.aggr = aggr
        self.weight = nn.Parameter(torch.Tensor(emb_dim, emb_dim))
        self.bias = nn.Parameter(torch.Tensor(emb_dim))
        stdv = math.sqrt(6.0 / (emb_dim + emb_dim))
        self.weight.data.uniform_(-stdv, stdv)
        self.bias.data.fill_(0)
        self.edge_embedding1 = nn.Embedding(_NUM_BOND_TYPE, 1)
        self.edge_embedding2 = nn.Embedding(_NUM_BOND_DIRECTION, 1)
        nn.init.xavier_uniform_(self.edge_embedding1.weight.data)
        nn.init.xavier_uniform_(self.edge_embedding2.weight.data)

    def forward(self, x, edge_index, edge_attr):
        edge_index = add_self_loops(edge_index, num_nodes=x.size(0))[0]
        self_loop_attr = torch.zeros(x.size(0), 2, device=edge_attr.device, dtype=edge_attr.dtype)
        self_loop_attr[:, 0] = 4
        edge_attr = torch.cat((edge_attr, self_loop_attr), dim=0)
        edge_embeddings = self.edge_embedding1(edge_attr[:, 0]) + self.edge_embedding2(edge_attr[:, 1])
        edge_index, _ = _gcn_norm(edge_index)
        x = x @ self.weight
        return self.propagate(edge_index, x=x, edge_attr=edge_embeddings, size=None) + self.bias

    def message(self, x_j, edge_attr):
        return x_j if edge_attr is None else edge_attr + x_j

    def message_and_aggregate(self, adj_t, x):
        return torch_sparse.matmul(adj_t, x, reduce=self.aggr)


class GCN(nn.Module):
    def __init__(self, num_layer, emb_dim, feat_dim, drop_ratio):
        super().__init__()
        self.num_layer = num_layer
        self.drop_ratio = drop_ratio
        self.x_embedding1 = nn.Embedding(_NUM_ATOM_TYPE, emb_dim)
        self.x_embedding2 = nn.Embedding(_NUM_CHIRALITY_TAG, emb_dim)
        nn.init.xavier_uniform_(self.x_embedding1.weight.data)
        nn.init.xavier_uniform_(self.x_embedding2.weight.data)
        self.gnns = nn.ModuleList([GCNConv(emb_dim) for _ in range(num_layer)])
        self.batch_norms = nn.ModuleList([nn.BatchNorm1d(emb_dim) for _ in range(num_layer)])
        self.feat_lin = nn.Linear(emb_dim, feat_dim)

    def forward(self, x, edge_index, edge_attr, batch):
        from torch_geometric.nn import global_mean_pool
        h = self.x_embedding1(x[:, 0]) + self.x_embedding2(x[:, 1])
        for i in range(self.num_layer):
            h = self.gnns[i](h, edge_index, edge_attr)
            h = self.batch_norms[i](h)
            h = F.dropout(h if i == self.num_layer - 1 else F.relu(h), 
                         self.drop_ratio, training=self.training)
        return self.feat_lin(global_mean_pool(h, batch))

class CombinedModel(nn.Module):
    def __init__(self, transformer, gcn):
        super().__init__()
        self.transformer = transformer
        self.gcn = gcn
    
    def forward(self, token_idx, x, edge_index, edge_attr, batch):
        h_transformer = self.transformer(token_idx)
        h_gcn = self.gcn(x, edge_index, edge_attr, batch)
        return torch.cat([h_transformer, h_gcn], dim=1)

class Predictor(nn.Module):
    """ESOL predictor: Linear -> Dropout -> ReLU -> Linear"""
    def __init__(self, in_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, in_dim * 2),
            nn.Dropout(p=dropout),
            nn.ReLU(),
            nn.Linear(in_dim * 2, 1),
        )
    
    def forward(self, x):
        return self.net(x)

class FullModel(nn.Module):
    def __init__(self, combined, predictor):
        super().__init__()
        self.combined = combined
        self.predictor = predictor
    
    def forward(self, token_idx, x, edge_index, edge_attr, batch):
        combined_out = self.combined(token_idx, x, edge_index, edge_attr, batch)
        return self.predictor(combined_out)

# ============================================================
# TRAINING
# ============================================================

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for batch in loader:
        tokens_idx, atom_mask, targets, x, edge_index, edge_attr, batch_idx = batch
        tokens_idx = tokens_idx.to(device)
        targets = targets.to(device)
        x = x.to(device)
        edge_index = edge_index.to(device)
        edge_attr = edge_attr.to(device)
        batch_idx = batch_idx.to(device)
        
        optimizer.zero_grad()
        outputs = model(tokens_idx, x, edge_index, edge_attr, batch_idx)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * tokens_idx.size(0)
    
    return total_loss / len(loader.dataset)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    predictions, actuals = [], []
    
    with torch.no_grad():
        for batch in loader:
            tokens_idx, atom_mask, targets, x, edge_index, edge_attr, batch_idx = batch
            tokens_idx = tokens_idx.to(device)
            targets = targets.to(device)
            x = x.to(device)
            edge_index = edge_index.to(device)
            edge_attr = edge_attr.to(device)
            batch_idx = batch_idx.to(device)
            
            outputs = model(tokens_idx, x, edge_index, edge_attr, batch_idx)
            loss = criterion(outputs, targets)
            
            total_loss += loss.item() * tokens_idx.size(0)
            predictions.extend(outputs.cpu().numpy())
            actuals.extend(targets.cpu().numpy())
    
    avg_loss = total_loss / len(loader.dataset)
    rmse = np.sqrt(np.mean((np.array(predictions) - np.array(actuals)) ** 2))
    
    return avg_loss, rmse

# ============================================================
# MAIN
# ============================================================

def main():
    print("Loading dataset...")
    
    # Check if CSV exists
    if not os.path.exists(ESOL_CSV_PATH):
        print(f"ERROR: ESOL CSV not found at {ESOL_CSV_PATH}")
        print("Please update ESOL_CSV_PATH in the script.")
        return
    
    # Load dataset
    dataset = ESOLDataset(ESOL_CSV_PATH)
    
    # Split: 843 train, 96 val, 96 test (matching original)
    train_size = 843
    val_size = 96
    test_size = len(dataset) - train_size - val_size
    
    train_dataset, temp_dataset = torch.utils.data.random_split(
        dataset, [train_size, len(dataset) - train_size],
        generator=torch.Generator().manual_seed(SEED)
    )
    val_dataset, test_dataset = torch.utils.data.random_split(
        temp_dataset, [val_size, test_size],
        generator=torch.Generator().manual_seed(SEED)
    )
    
    print(f"Split: train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}")
    
    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
    
    # Build model
    print("\nBuilding model...")
    transformer = TransformerEncoder(
        D_MODEL, N_LAYERS, VOCAB_SIZE, MAXLEN, D_K, D_V, N_HEADS, D_FF,
        bias_init=ELECTRON_BIAS_INIT,
        bias_learnable=ELECTRON_BIAS_LEARNABLE
    )
    gcn = GCN(GCN_LAYERS, GCN_EMB_DIM, GCN_FEAT_DIM, drop_ratio=0.1)
    combined = CombinedModel(transformer, gcn)
    predictor = Predictor(COMBINED_DIM, dropout=PREDICTOR_DROPOUT)
    model = FullModel(combined, predictor).to(device)
    
    # Optimizer and scheduler
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=LR_FACTOR, patience=LR_PATIENCE, min_lr=LR_MIN
    )
    criterion = nn.MSELoss()
    
    # Training loop
    print(f"\nTraining for up to {EPOCHS} epochs (early stop patience={EARLY_STOP_PATIENCE})...\n")
    
    best_val_rmse = float('inf')
    best_epoch = 0
    best_test_rmse = 0
    patience_counter = 0
    start_time = time.time()
    
    for epoch in range(1, EPOCHS + 1):
        epoch_start = time.time()
        
        # Train
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        
        # Evaluate
        val_loss, val_rmse = evaluate(model, val_loader, criterion, device)
        test_loss, test_rmse = evaluate(model, test_loader, criterion, device)
        
        # Scheduler step
        scheduler.step(val_rmse)
        current_lr = optimizer.param_groups[0]['lr']
        
        # Get electron_bias value
        e_bias_value = model.combined.transformer.embedding.electron_bias.item()
        
        epoch_time = time.time() - epoch_start
        
        print(f"epoch {epoch:03d}/{EPOCHS} | loss: {train_loss:.2f} | "
              f"val_rmse: {val_rmse:.5f} | test_rmse: {test_rmse:.5f} | "
              f"lr: {current_lr:.2e} | e_bias: {e_bias_value:.5f} | t: {epoch_time:.1f}s")
        
        # Early stopping
        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_test_rmse = test_rmse
            best_epoch = epoch
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\nEarly stopping: no improvement for {EARLY_STOP_PATIENCE} epochs.")
            break
    
    total_time = (time.time() - start_time) / 60
    
    # Final results
    print(f"\n{'='*64}")
    print(f"FINAL RESULTS - {EXPERIMENT_NAME}")
    print(f"{'='*64}")
    print(f"Best epoch         : {best_epoch}")
    print(f"Best val RMSE      : {best_val_rmse:.5f}")
    print(f"Best test RMSE     : {best_test_rmse:.5f}")
    print(f"Learned e_bias     : {e_bias_value:.6f}")
    print(f"Training time      : {total_time:.1f} min")
    print(f"{'='*64}\n")
    
    # Comparison table
    print("COMPARISON WITH OTHER VARIANTS:")
    print(f"{'='*64}")
    print(f"{'Variant':<30} {'Test RMSE':<15} {'Improvement'}")
    print(f"{'-'*64}")
    print(f"{'0: Baseline (bias, 5 layers)':<30} {'0.48509':<15} {'-'}")
    print(f"{'1: No Bias (5 layers)':<30} {'0.47483':<15} {'2.1% better'}")
    print(f"{'4: Arch Tuned (bias, 3 layers)':<30} {'0.46546':<15} {'4.0% better'}")
    print(f"{'5: No-Bias + Arch Tuned':<30} {f'{best_test_rmse:.5f}':<15} ", end='')
    
    improvement_vs_baseline = (0.48509 - best_test_rmse) / 0.48509 * 100
    improvement_vs_arch_tuned = (0.46546 - best_test_rmse) / 0.46546 * 100
    
    print(f"{improvement_vs_baseline:.1f}% vs baseline")
    print(f"{'':30} {'':15} {improvement_vs_arch_tuned:.1f}% vs arch tuned")
    print(f"{'='*64}\n")
    
    # Conclusion
    if best_test_rmse < 0.46546:
        print("CONCLUSION: No-bias + arch tuned BEATS arch tuned with bias.")
        print("→ The electron_bias parameter HURTS performance even with optimal architecture.")
        print("→ Recommendation: Remove electron_bias from the model.")
    elif best_test_rmse > 0.46546:
        print("CONCLUSION: Arch tuned with bias BEATS no-bias + arch tuned.")
        print("→ The electron_bias parameter HELPS when architecture is properly configured.")
        print("→ Recommendation: Keep electron_bias and use tuned architecture.")
    else:
        print("CONCLUSION: No-bias + arch tuned EQUALS arch tuned with bias.")
        print("→ The electron_bias parameter makes NO DIFFERENCE.")
        print("→ Recommendation: Remove electron_bias for simplicity (Occam's razor).")

if __name__ == '__main__':
    main()

Device: cuda

Variant 5: No-Bias + Arch Tuned
Configuration:
  electron_bias_init : 0.0
  electron_bias_learnable: False
  gcn_layers         : 3
  predictor_dropout  : 0.2

Loading dataset...
Loaded 1035 valid molecules
Split: train=843, val=96, test=96

Building model...

Training for up to 100 epochs (early stop patience=50)...

epoch 001/100 | loss: 2.65 | val_rmse: 1.47189 | test_rmse: 1.43154 | lr: 2.00e-05 | e_bias: 0.00000 | t: 54.4s
epoch 002/100 | loss: 1.30 | val_rmse: 0.89659 | test_rmse: 0.83735 | lr: 2.00e-05 | e_bias: 0.00000 | t: 56.7s
epoch 003/100 | loss: 0.65 | val_rmse: 0.70714 | test_rmse: 0.71551 | lr: 2.00e-05 | e_bias: 0.00000 | t: 59.8s
epoch 004/100 | loss: 0.44 | val_rmse: 0.65860 | test_rmse: 0.67876 | lr: 2.00e-05 | e_bias: 0.00000 | t: 59.2s
epoch 005/100 | loss: 0.36 | val_rmse: 0.59422 | test_rmse: 0.64122 | lr: 2.00e-05 | e_bias: 0.00000 | t: 59.1s
epoch 006/100 | loss: 0.37 | val_rmse: 0.60067 | test_rmse: 0.61968 | lr: 2.00e-05 | e_bias: 0.00000 | t: 